# Fink/LSST — Detection Threshold vs. Focal-Plane Radius

This notebook reads the src Parquet files saved by `01_fink_block_lightcurves.ipynb`
and produces:

1. **All-band focal-plane heatmap** (1×2): median psfFlux and median magnitude
   aggregated over all bands together (global overview only).

2. **Per-band focal-plane heatmap** (6×2 grid): one row per band `u g r i z y`,
   left column = median psfFlux, right column = median magnitude.
   - Each row has its own independent colour scale (band isolation).
   - A single **horizontal colour bar** is shared within each column and placed
     at the bottom of the figure.

3. **Per-band error plots** (6×2 grid): median psfFlux (left) and median
   magnitude (right) vs. focal-plane radius $r$, one row per band.
   - Same shared X axis; independent Y axis per row.
   - One horizontal colour bar per column at the bottom.

### Motivation

The median PSF flux of *detected* src alerts is a proxy for the **per-visit
detection threshold**: a higher median flux means fainter sources are not
detected.  The radial profile reveals whether the threshold worsens at large
focal-plane radii (PSF degradation, background rise).

No functional fit is applied here — the goal is to **visualise the shape** of
these radial profiles before choosing a fitting function.

### Per-CCD uncertainty

$$\sigma_{\rm med} \approx 1.4826 \times \frac{\mathrm{MAD}}{\sqrt{N}}$$

---

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS
- **Creation date:** 2026-04-03
- **Last update:** 2026-04-04
- **Subject:** Fink alert broker — Rubin LSST detection threshold vs. focal-plane radius

## 1. Imports & configuration

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = "figs_FINK_BLOCK_LC_05c"  # output figures
os.makedirs(DIR_FIGS, exist_ok=True)

PATH_GEOM_CSV = "data_FocalPlane/ccd_geometry.csv"

GROUPS_OF_INTEREST = ["gaia_star_stable", "gaia_star_stable_hq"]

COL_DETECTOR = "r:detector"
COL_PSF = "r:psfFlux"  # [nJy]
COL_PSFERR = "r:psfFluxErr"  # [nJy]
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"

AB_ZP = 31.4  # AB zero-point for nJy
MAD_SCALE = 1.4826  # MAD → sigma for a Gaussian

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BANDS = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Geometry CSV     : {os.path.abspath(PATH_GEOM_CSV)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag(flux_nJy, zero_point=AB_ZP):
    """Convert PSF flux [nJy] to AB magnitude; non-positive flux → NaN."""
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f) + zero_point, np.nan)


def mad_std(x):
    """Robust sigma via MAD: 1.4826 * median(|x - median(x)|)."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    return MAD_SCALE * float(np.median(np.abs(x - np.median(x))))


def median_error(x):
    """Standard error on the median: 1.4826 * MAD / sqrt(N)."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 2:
        return np.nan
    return mad_std(x) / np.sqrt(n)


print("Utility functions defined: flux_nJy_to_mag, mad_std, median_error.")

## 3. Load src Parquet files and concatenate

In [ ]:
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path):
    return os.path.basename(path).replace("_src.parquet", "")


groups_src = {group_from_path(p): p for p in src_files}
print(f"src parquet files found: {len(groups_src)}")
for g in groups_src:
    print(f"  {g}")

In [ ]:
dfs_to_concat = []
for group in GROUPS_OF_INTEREST:
    if group not in groups_src:
        print(f"  WARNING: group '{group}' not found — skipped.")
        continue
    df_g = pd.read_parquet(groups_src[group])
    df_g["_group"] = group
    dfs_to_concat.append(df_g)
    print(f"  Loaded {group}: {len(df_g):,} rows")

if not dfs_to_concat:
    raise RuntimeError(f"None of {GROUPS_OF_INTEREST} found in {DIR_DATA}.")

df_src = pd.concat(dfs_to_concat, ignore_index=True)
print(f"\nTotal src rows: {len(df_src):,}")

## 4. Schema inspection & column validation

In [ ]:
print("Columns in src table:")
print(df_src.dtypes.to_string())

In [ ]:
required = [COL_DETECTOR, COL_PSF, COL_PSFERR]
missing = [c for c in required if c not in df_src.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")
print("All required columns present.")
print(f"Unique CCDs : {df_src[COL_DETECTOR].nunique()}")

## 5. Compute magnitude column

$$m = -2.5 \log_{10}(F_{\rm psf}\,[{\rm nJy}]) + 31.4$$

In [ ]:
df_src = df_src.copy()
valid_flux = (df_src[COL_PSF].values > 0) & np.isfinite(df_src[COL_PSF].values)
flux_vals = df_src[COL_PSF].values.astype(float)
flux_vals[~valid_flux] = np.nan
df_src["psfFlux"] = flux_vals
df_src["mag"] = flux_nJy_to_mag(df_src["psfFlux"].values)

n_valid = (~np.isnan(df_src["psfFlux"])).sum()
n_invalid = np.isnan(df_src["psfFlux"]).sum()
print(f"psfFlux valid   : {n_valid:,}")
print(f"psfFlux masked  : {n_invalid:,}  (non-positive → NaN)")
print(f"psfFlux range   : [{np.nanmin(df_src['psfFlux']):.2f},  {np.nanmax(df_src['psfFlux']):.2f}] nJy")
print(f"mag range       : [{np.nanmin(df_src['mag']):.2f},  {np.nanmax(df_src['mag']):.2f}] mag")

## 6. All-band per-CCD aggregation

In [ ]:
df_valid = df_src.dropna(subset=["psfFlux", "mag"])

ccd_stats = (
    df_valid.groupby(COL_DETECTOR)
    .agg(
        n_src=("psfFlux", "count"),
        flux_med=("psfFlux", "median"),
        flux_err=("psfFlux", lambda x: median_error(x)),
        mag_med=("mag", "median"),
        mag_err=("mag", lambda x: median_error(x)),
    )
    .reset_index()
    .rename(columns={COL_DETECTOR: "detector"})
)
print(f"CCDs with valid src (all bands): {len(ccd_stats)}")

## 7. Load focal-plane geometry and merge

In [ ]:
if not os.path.exists(PATH_GEOM_CSV):
    raise FileNotFoundError(f"Geometry CSV not found: {PATH_GEOM_CSV}")

geom = pd.read_csv(PATH_GEOM_CSV)
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# Focal-plane bounding box (used by all heatmap functions)
FP_XMIN = geom[[f"corner{i}_x" for i in range(4)]].min().min()
FP_XMAX = geom[[f"corner{i}_x" for i in range(4)]].max().max()
FP_YMIN = geom[[f"corner{i}_y" for i in range(4)]].min().min()
FP_YMAX = geom[[f"corner{i}_y" for i in range(4)]].max().max()

geom_stats = geom.merge(
    ccd_stats[["detector", "n_src", "flux_med", "flux_err", "mag_med", "mag_err"]],
    on="detector",
    how="left",
)
print(f"Geometry loaded : {len(geom)} CCDs")
print(f"CCDs with data  : {geom_stats['flux_med'].notna().sum()}")
print(f"CCDs without data: {geom_stats['flux_med'].isna().sum()}")

## 8. Helper functions

In [ ]:
def draw_focal_plane_panel(ax, geom_df, value_col, cmap_obj, norm, title, show_ccd_label=True):
    """
    Draw one focal-plane heatmap panel on *ax* — no individual colourbar.
    The caller is responsible for adding a shared colourbar.

    Parameters
    ----------
    ax           : Axes
    geom_df      : pd.DataFrame — geometry + statistics
    value_col    : str          — column to colour-code
    cmap_obj     : colormap
    norm         : Normalize
    title        : str
    show_ccd_label : bool
    """
    for _, row in geom_df.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row[value_col]
        face_color = "lightgrey" if np.isnan(val) else cmap_obj(norm(val))
        poly = patches.Polygon(corners, facecolor=face_color, edgecolor="black", linewidth=0.2)
        ax.add_patch(poly)
        if show_ccd_label:
            ax.text(
                row["x_center"],
                row["y_center"],
                str(int(row["detector"])),
                ha="center",
                va="center",
                fontsize=5,
                fontweight="bold",
                color="k",
            )

    ax.set_aspect("equal")
    ax.set_xlim(FP_XMIN, FP_XMAX)
    ax.set_ylim(FP_YMIN, FP_YMAX)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=9)


def build_band_geom_stats(df_src_full, band, geom_df):
    """
    Compute per-CCD median flux and magnitude for a single photometric band
    and merge with the focal-plane geometry.  Band isolation guaranteed.

    Returns
    -------
    geom_band : pd.DataFrame
    n_src     : int
    """
    df_b = df_src_full[df_src_full[COL_BAND] == band].dropna(subset=["psfFlux", "mag"])
    if len(df_b) == 0:
        geom_band = geom_df.copy()
        for col in ("n_src", "flux_med", "flux_err", "mag_med", "mag_err"):
            geom_band[col] = np.nan
        return geom_band, 0

    band_stats = (
        df_b.groupby(COL_DETECTOR)
        .agg(
            n_src=("psfFlux", "count"),
            flux_med=("psfFlux", "median"),
            flux_err=("psfFlux", lambda x: median_error(x)),
            mag_med=("mag", "median"),
            mag_err=("mag", lambda x: median_error(x)),
        )
        .reset_index()
        .rename(columns={COL_DETECTOR: "detector"})
    )
    return geom_df.merge(band_stats, on="detector", how="left"), int(len(df_b))


print("draw_focal_plane_panel() and build_band_geom_stats() defined.")

## 9. All-band heatmap (1×2)

Global overview — all bands mixed.  Not physically meaningful for flux
comparison across bands.  See Section 10 for the scientifically correct
per-band version.

In [ ]:
flux_vals_ccd = geom_stats["flux_med"].dropna()
mag_vals_ccd = geom_stats["mag_med"].dropna()

vmin_flux = float(np.percentile(flux_vals_ccd, 5)) if len(flux_vals_ccd) > 0 else 0
vmax_flux = float(np.percentile(flux_vals_ccd, 95)) if len(flux_vals_ccd) > 0 else 1
vmin_mag = float(np.percentile(mag_vals_ccd, 5)) if len(mag_vals_ccd) > 0 else 15
vmax_mag = float(np.percentile(mag_vals_ccd, 95)) if len(mag_vals_ccd) > 0 else 24

cmap_flux = plt.get_cmap("plasma")
cmap_mag = plt.get_cmap("plasma_r")
norm_flux_all = mcolors.Normalize(vmin=vmin_flux, vmax=vmax_flux)
norm_mag_all = mcolors.Normalize(vmin=vmin_mag, vmax=vmax_mag)

fig, (ax_f, ax_m) = plt.subplots(1, 2, figsize=(18, 9), constrained_layout=True)

draw_focal_plane_panel(
    ax_f, geom_stats, "flux_med", cmap_flux, norm_flux_all, r"All bands — Median $F_{\rm psf}$ [nJy]"
)
draw_focal_plane_panel(ax_m, geom_stats, "mag_med", cmap_mag, norm_mag_all, r"All bands — Median $m$ [mag]")

for ax, cmap_obj, norm, label in [
    (ax_f, cmap_flux, norm_flux_all, r"median $F_{\rm psf}$ [nJy]"),
    (ax_m, cmap_mag, norm_mag_all, r"median $m$ [mag]"),
]:
    sm = ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label=label)

fig.suptitle(
    r"LSSTCam Focal Plane — Detection threshold (all bands combined, global overview)  "
    f"groups: {', '.join(GROUPS_OF_INTEREST)}",
    fontsize=11,
)
savefig("focal_plane_FluxThreshold_allbands_combined")
plt.show()

## 10. Per-band focal-plane heatmap — 6×2 grid

Each row corresponds to one photometric band `u g r i z y`.
- **Left column**: median psfFlux [nJy] — **each row has its own colour scale**.
- **Right column**: median magnitude [mag] — **each row has its own colour scale**.
- Flux bands are never mixed.
- A **horizontal colour bar per column** is placed at the bottom of the figure,
  showing the range of the **last active band** as a reference.  Individual
  per-row scales are printed in the subplot title.
- Grey patches = CCDs with no detection in that band.

In [ ]:
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping per-band heatmaps.")
else:
    # ── Pre-compute per-band geometry + colour scales ──────────────────────────
    band_geom = {}  # band -> (geom_df, n_src)
    band_norms_flux = {}  # band -> Normalize for flux
    band_norms_mag = {}  # band -> Normalize for mag

    for band in BANDS:
        geom_b, n_src_b = build_band_geom_stats(df_src, band, geom)
        band_geom[band] = (geom_b, n_src_b)

        flux_b = geom_b["flux_med"].dropna()
        mag_b = geom_b["mag_med"].dropna()

        vmin_f = float(np.percentile(flux_b, 5)) if len(flux_b) > 0 else 0
        vmax_f = float(np.percentile(flux_b, 95)) if len(flux_b) > 0 else 1
        vmin_m = float(np.percentile(mag_b, 5)) if len(mag_b) > 0 else 15
        vmax_m = float(np.percentile(mag_b, 95)) if len(mag_b) > 0 else 24
        if vmax_f <= vmin_f:
            vmax_f = vmin_f + 1.0
        if vmax_m <= vmin_m:
            vmax_m = vmin_m + 0.5

        band_norms_flux[band] = mcolors.Normalize(vmin=vmin_f, vmax=vmax_f)
        band_norms_mag[band] = mcolors.Normalize(vmin=vmin_m, vmax=vmax_m)

    active_bands = [b for b in BANDS if band_geom[b][1] > 0]
    n_rows = len(active_bands)
    print(f"Active bands: {active_bands}")

    # ── Build the 6×2 figure ───────────────────────────────────────────────────
    # Reserve space for two horizontal colour bars at the bottom.
    # We use gridspec_kw to add bottom padding via subplot_adjust later.
    fig_hm, axes_hm = plt.subplots(
        n_rows,
        2,
        figsize=(16, 6.0 * n_rows + 1.2),
    )  # +1.2 inch for colour bars
    # layout="constrained")
    plt.subplots_adjust(left=0.03, right=0.97, top=0.94, bottom=0.10, hspace=0.35, wspace=0.08)

    if n_rows == 1:
        axes_hm = axes_hm[np.newaxis, :]

    for row_idx, band in enumerate(active_bands):
        geom_b, n_src_b = band_geom[band]
        norm_f = band_norms_flux[band]
        norm_m = band_norms_mag[band]
        bcolor = BAND_COLORS[band]

        vf0, vf1 = norm_f.vmin, norm_f.vmax
        vm0, vm1 = norm_m.vmin, norm_m.vmax

        draw_focal_plane_panel(
            axes_hm[row_idx, 0],
            geom_b,
            "flux_med",
            plt.get_cmap("plasma"),
            norm_f,
            rf"Band {band} — $F_{{\rm psf}}$ [{vf0:.0f}–{vf1:.0f}] nJy  (N={n_src_b:,})",
        )
        draw_focal_plane_panel(
            axes_hm[row_idx, 1],
            geom_b,
            "mag_med",
            plt.get_cmap("plasma_r"),
            norm_m,
            rf"Band {band} — $m$ [{vm0:.2f}–{vm1:.2f}] mag  (N={n_src_b:,})",
        )
        # Colour the band label
        axes_hm[row_idx, 0].set_ylabel(
            band,
            fontsize=13,
            fontweight="bold",
            color=bcolor,
            rotation=0,
            labelpad=14,
            va="center",
        )

    # ── Two horizontal colour bars at the bottom ───────────────────────────────
    # For the flux column we use the last active band's scale as reference.
    last = active_bands[-1]

    # Flux colour bar — centred below the left column
    ax_cb_flux = fig_hm.add_axes([0.06, 0.02, 0.38, 0.025])  # [left, bottom, w, h]
    sm_flux = ScalarMappable(cmap=plt.get_cmap("plasma"), norm=band_norms_flux[last])
    sm_flux.set_array([])
    cb_flux = fig_hm.colorbar(sm_flux, cax=ax_cb_flux, orientation="horizontal")
    cb_flux.set_label(
        rf"Median $F_{{\rm psf}}$ [nJy]   (colour scale = band {last}; "
        r"each row is independent)",
        fontsize=8,
    )

    # Magnitude colour bar — centred below the right column
    ax_cb_mag = fig_hm.add_axes([0.56, 0.02, 0.38, 0.025])
    sm_mag = ScalarMappable(cmap=plt.get_cmap("plasma_r"), norm=band_norms_mag[last])
    sm_mag.set_array([])
    cb_mag = fig_hm.colorbar(sm_mag, cax=ax_cb_mag, orientation="horizontal")
    cb_mag.set_label(
        rf"Median $m$ [mag]   (colour scale = band {last}; "
        r"each row is independent)",
        fontsize=8,
    )

    fig_hm.suptitle(
        r"LSSTCam Focal Plane — Detection threshold per band  "
        r"(left: median $F_{\rm psf}$ / right: median $m$  "
        r"— each row: independent band-only colour scale)",
        fontsize=12,
        y=0.97,
    )

    savefig("focal_plane_FluxThreshold_per_band_6x2")
    plt.show()

## 11. All-band error plot (1×2)

Global radial profile — all bands mixed.  Each point is one CCD; error bars
represent $\sigma_{\rm med} \approx 1.4826 \times {\rm MAD}/\sqrt{N}$.

In [ ]:
geom_plot = geom_stats.dropna(subset=["flux_med", "mag_med"]).copy()
print(f"CCDs used in all-band radial plots: {len(geom_plot)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), constrained_layout=True)

sc1 = ax1.scatter(
    geom_plot["r_mm"], geom_plot["flux_med"], c=geom_plot["n_src"], cmap="viridis", s=40, zorder=4
)
ax1.errorbar(
    geom_plot["r_mm"],
    geom_plot["flux_med"],
    yerr=geom_plot["flux_err"],
    fmt="none",
    ecolor="grey",
    elinewidth=0.8,
    capsize=2,
    alpha=0.6,
    zorder=3,
)
fig.colorbar(sc1, ax=ax1, fraction=0.03, pad=0.01).set_label("N src / CCD", fontsize=8)
ax1.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
ax1.set_ylabel(r"Median $F_{\rm psf}$  [nJy]", fontsize=10)
ax1.set_title(r"All bands — Median $F_{\rm psf}$ vs. $r$", fontsize=10)
ax1.set_xlim(-1, geom_plot["r_mm"].max() * 1.05)
ax1.set_ylim(bottom=0)

sc2 = ax2.scatter(
    geom_plot["r_mm"], geom_plot["mag_med"], c=geom_plot["n_src"], cmap="viridis", s=40, zorder=4
)
ax2.errorbar(
    geom_plot["r_mm"],
    geom_plot["mag_med"],
    yerr=geom_plot["mag_err"],
    fmt="none",
    ecolor="grey",
    elinewidth=0.8,
    capsize=2,
    alpha=0.6,
    zorder=3,
)
fig.colorbar(sc2, ax=ax2, fraction=0.03, pad=0.01).set_label("N src / CCD", fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=10)
ax2.set_ylabel(r"Median $m$  [mag]", fontsize=10)
ax2.set_title(r"All bands — Median $m$ vs. $r$", fontsize=10)
ax2.set_xlim(-1, geom_plot["r_mm"].max() * 1.05)

fig.suptitle(
    r"Detection threshold — all bands combined  "
    f"(groups: {', '.join(GROUPS_OF_INTEREST)},  N_CCD={len(geom_plot)})",
    fontsize=11,
)
savefig("FluxThreshold_flux_and_mag_vs_radius_allbands")
plt.show()

## 12. Per-band error plots — 6×2 grid

One row per band `u g r i z y`.  Left column = median psfFlux vs. $r$;
right column = median magnitude vs. $r$.

- **Shared X axis** across all rows (same radial range).
- **Independent Y axis** per row (each band has its own flux/magnitude range).
- Each subplot is coloured with the band's canonical colour.
- Two horizontal colour bars at the bottom of the figure (one per column)
  show the N_src scale (viridis, last active band as reference).

In [ ]:
if COL_BAND not in df_src.columns:
    print(f"Column '{COL_BAND}' not found — skipping per-band radial plots.")
else:
    # ── Pre-compute per-band radial data ──────────────────────────────────────
    band_radial = {}  # band -> geom_band DataFrame (with r_mm, flux_med, ...)

    for band in BANDS:
        df_band = df_src[df_src[COL_BAND] == band].dropna(subset=["psfFlux", "mag"])
        if len(df_band) == 0:
            continue
        band_stats = (
            df_band.groupby(COL_DETECTOR)
            .agg(
                n_src=("psfFlux", "count"),
                flux_med=("psfFlux", "median"),
                flux_err=("psfFlux", lambda x: median_error(x)),
                mag_med=("mag", "median"),
                mag_err=("mag", lambda x: median_error(x)),
            )
            .reset_index()
            .rename(columns={COL_DETECTOR: "detector"})
        )
        geom_band = geom[["detector", "r_mm"]].merge(band_stats, on="detector", how="inner")
        if len(geom_band) > 0:
            band_radial[band] = geom_band

    active_bands_r = [b for b in BANDS if b in band_radial]
    n_rows_r = len(active_bands_r)
    print(f"Active bands for radial plots: {active_bands_r}")

    # ── Global X range (same for all rows) ────────────────────────────────────
    r_max_global = max(band_radial[b]["r_mm"].max() for b in active_bands_r) * 1.05

    # ── Global N_src range for the colour bar ─────────────────────────────────
    nsrc_min = min(band_radial[b]["n_src"].min() for b in active_bands_r)
    nsrc_max = max(band_radial[b]["n_src"].max() for b in active_bands_r)
    norm_nsrc = mcolors.Normalize(vmin=nsrc_min, vmax=nsrc_max)
    cmap_nsrc = plt.get_cmap("viridis")

    # ── Build the 6×2 figure ──────────────────────────────────────────────────
    fig_ep, axes_ep = plt.subplots(
        n_rows_r,
        2,
        figsize=(14, 3.5 * n_rows_r + 1.2),
        sharex=True,  # shared X axis
        sharey=False,  # independent Y axis per row
    )
    plt.subplots_adjust(left=0.09, right=0.97, top=0.94, bottom=0.12, hspace=0.45, wspace=0.30)

    if n_rows_r == 1:
        axes_ep = axes_ep[np.newaxis, :]

    for row_idx, band in enumerate(active_bands_r):
        gb = band_radial[band]
        bc = BAND_COLORS[band]
        ax_f = axes_ep[row_idx, 0]
        ax_m = axes_ep[row_idx, 1]
        n_total = int(gb["n_src"].sum())
        n_ccd = len(gb)

        # ── Flux panel ────────────────────────────────────────────────────────
        sc = ax_f.scatter(
            gb["r_mm"], gb["flux_med"], c=gb["n_src"], cmap=cmap_nsrc, norm=norm_nsrc, s=30, zorder=4
        )
        ax_f.errorbar(
            gb["r_mm"],
            gb["flux_med"],
            yerr=gb["flux_err"],
            fmt="none",
            ecolor=bc,
            elinewidth=0.7,
            capsize=1.5,
            alpha=0.55,
            zorder=3,
        )
        ax_f.set_ylabel(rf"$F_{{\rm psf}}$ [nJy]", fontsize=8)
        ax_f.set_title(
            rf"Band {band} — $F_{{\rm psf}}$ vs $r$  (N_CCD={n_ccd}, N_src={n_total:,})",
            fontsize=9,
            color=bc,
            fontweight="bold",
        )
        ax_f.set_ylim(bottom=0)
        ax_f.set_xlim(-1, r_max_global)
        if row_idx == n_rows_r - 1:
            ax_f.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=9)

        # ── Magnitude panel ───────────────────────────────────────────────────
        ax_m.scatter(gb["r_mm"], gb["mag_med"], c=gb["n_src"], cmap=cmap_nsrc, norm=norm_nsrc, s=30, zorder=4)
        ax_m.errorbar(
            gb["r_mm"],
            gb["mag_med"],
            yerr=gb["mag_err"],
            fmt="none",
            ecolor=bc,
            elinewidth=0.7,
            capsize=1.5,
            alpha=0.55,
            zorder=3,
        )
        ax_m.invert_yaxis()
        ax_m.set_ylabel(r"$m$ [mag]", fontsize=8)
        ax_m.set_title(
            rf"Band {band} — $m$ vs $r$  (N_CCD={n_ccd}, N_src={n_total:,})",
            fontsize=9,
            color=bc,
            fontweight="bold",
        )
        ax_m.set_xlim(-1, r_max_global)
        if row_idx == n_rows_r - 1:
            ax_m.set_xlabel(r"$r = \sqrt{x^2+y^2}$  [mm]", fontsize=9)

    # ── Two horizontal colour bars at the bottom ──────────────────────────────
    # Both columns use the same N_src scale so we share one colourbar object
    # but place two axes for symmetry under each column.
    ax_cb_l = fig_ep.add_axes([0.09, 0.03, 0.38, 0.022])
    ax_cb_r = fig_ep.add_axes([0.57, 0.03, 0.38, 0.022])

    sm_nsrc = ScalarMappable(cmap=cmap_nsrc, norm=norm_nsrc)
    sm_nsrc.set_array([])
    for ax_cb, col_label in [
        (ax_cb_l, r"N src per CCD  (left column — $F_{\rm psf}$ vs $r$)"),
        (ax_cb_r, r"N src per CCD  (right column — $m$ vs $r$)"),
    ]:
        cb = fig_ep.colorbar(sm_nsrc, cax=ax_cb, orientation="horizontal")
        cb.set_label(col_label, fontsize=8)

    fig_ep.suptitle(
        r"Detection threshold vs. focal-plane radius — per band 6×2  "
        r"(left: median $F_{\rm psf}$ / right: median $m$  "
        r"— shared $r$ axis, independent $y$ axis)",
        fontsize=12,
        y=0.97,
    )

    savefig("FluxThreshold_flux_and_mag_vs_radius_per_band_6x2")
    plt.show()

## 13. Summary table: per-CCD detection threshold statistics (all bands)

In [ ]:
summary_cols = [
    "detector",
    "name",
    "x_center",
    "y_center",
    "r_mm",
    "n_src",
    "flux_med",
    "flux_err",
    "mag_med",
    "mag_err",
]
df_summary = geom_plot[summary_cols].sort_values("r_mm").reset_index(drop=True)
print("Per-CCD detection threshold summary — all bands — sorted by r:")
display(df_summary)

In [ ]:
summary_path = os.path.join(DIR_FIGS, "ccd_flux_threshold_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")